# Problem 1 -- Architecture critique


## Current Status of Ask Vantum Chat

### 1. Guidance Mode:
These two are running independently -- they do not have the shared data that helps the system consistent.

#### Endpoint with additional context: 
guidance mode uses current(general) chat endpoint with addtional guidanceContext:
- existing/general chat endpoint
- Kernel snapshot context

Added With:
- active task ID
- curent step ID
- full step plan

### 2. Proactive Observer Voice:

- /api/observer/step-greeting
- kernel snapshot context
- active task
- current step
- Greeting layer (stored in sessionStorage)

![My Image](p1-current-architecture.jpg)

## Problems

1.  LLM in each surface runs independently and they have fragmented memory/context that affects consistency. They two are isolated and separate mini agents that do not share the synced process, for example, what nudges fired, what guidance happended, what observer said and any context that should be shared and synced.
2.  Two layers use the same state/context (same kernel snapshot). Each one needs a pre-processed layer to receive specified states for its own

## Solution

![My Image](p1-solution.jpg)

### Key Point (IMPORTANT):  

Adding centralized reasoning layer where LLM generates output for both. And in this regards, the independent modules - Guidance Mode and Observer Voice act as a renderer that receives the LLM outputs from the Shared Observer Reasoning layer.

# Demo Code (Mockup)

## 1. Shared Observer Layer

In [ ]:
### Note: values aren't specified as prototype code

# Example memory/state
state = {
        "active_task": None, 
        "current_step": None,
        "guidance_history": [],
        "observer_history": []
}

class SharedObserverContext:
    def __init__(self, state): 
        self.state = state

    def update(self, key, value):
        self.state[key] = value

    def get_context(self):
        """
        Retrieve shared states 
        """
        return self.state

## 2. Shared Observer Reasoning

In [ ]:
context = SharedObserverReasoning(state)

class SharedObserverReasoning:
    def __init__(self, context):
        self.context = context

    def reason(self, obj):
        response = llm_reason(obj)
        return response

    def reason(self):
        active_task = self.reason(context["active_task"])
        current_step = self.reason(context["current_step"])

        return {
            "guidance": f"Focus on completing: {current_step}",
            "observer": f"You are currently progressing through: {active_task}"
        }

## 3. Renderer Layers
Guidance Mode & ProactiveObserver Voice

In [ ]:
class GuidanceMode: 
    
    def __init__(self, output):
        self.reasoning_output = output
        
    def render(self): 
        return self.reasoning_output["guidance"]



class ProactiveObserverVoice: 
    
    def __init__(self, output):
        self.reasoning_output = output
        
    def render(self): 
        return self.reasoning_output["observer"]